<a href="https://colab.research.google.com/github/nikitask14/pytorch-engineering-to-federated-learning/blob/main/Sitting20_Copying_models_safely_and_checkpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import copy

In [ ]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(2,4)
    self.layer2 = nn.Linear(4,1)

  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = self.layer2(x)

    return x


#####**Controlled starting conditions with torch.manual_seed()**
If I rerun this notebook tomorrow, how do I make sure my experiment starts from the same initial global model rather than a different random initialization?

Or simply, if we rerun the experiment, PyTorch can give the global model the same random starting weights, instead of a different initialization each time.

When we create:

**global_model = MyModel()**

PyTorch initializes the weights randomly. So rerunning the notebook can give the global model different initial weights.

In [ ]:
torch.manual_seed(42)
global_model = MyModel()
client1 = MyModel()
client2 = MyModel()

When we write:

global_model = MyModel()

PyTorch gives the model random starting weights.

So if we run the notebook again, we may get different starting weights.

If we write:

torch.manual_seed(42)
global_model = MyModel()

We are basically telling PyTorch:

“Use the same random sequence as before.”

So every time we rerun that code, the model starts with the same random weights.

In [ ]:
# extract the global model’s state into a variable
# load that state into client1 and client2.
global_model_state = global_model.state_dict()
client1.load_state_dict(global_model_state)
client2.load_state_dict(global_model_state)


<All keys matched successfully>

**Independent/Different model objects**


client1 and client2 must be different Python objects, so training one does not change the other.

In [ ]:
# verify the object independence
print(client1 is global_model)
print(client2 is client1)
print(client2 is global_model)

False
False
False


**Same/Identical starting parameter values**


Corresponding parameters must initially contain the same numerical values

In [ ]:
torch.equal(client1.state_dict()["layer1.weight"], client2.state_dict()["layer1.weight"])

True